# Day 27 — Project: Transfer Learning on Custom Data

## 1. Learning Objectives
- Synthesize all knowledge from Phase 4 into a state-of-the-art Computer Vision pipeline.
- Set up distinct Data Augmentation pipelines for Train vs Val.
- Load images using `torchvision.datasets.ImageFolder`.
- Fine-tune a Pre-trained `ResNet18` model.
- Train on GPU and evaluate performance.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
import torchvision.datasets as datasets
from torchvision.transforms import v2
import os

## Step 1: Device Configuration

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 2: Data Augmentation Pipelines
We must define two separate pipelines. Since we are using a pre-trained ResNet, we **must** resize our images to `224x224` and normalize them using the ImageNet statistics.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# 1. Train Transforms (Heavy Augmentation)
train_transforms = v2.Compose([
    v2.RandomResizedCrop(224),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# 2. Val/Test Transforms (NO Augmentation, just Resize, Crop, Tensor, Normalize)
val_transforms = v2.Compose([
    v2.Resize(256),            # Resize slightly larger
    v2.CenterCrop(224),        # Crop the exact center to 224x224
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Normalize(mean=imagenet_mean, std=imagenet_std)
])

## Step 3: Dataset & DataLoader
In a real scenario, you would have a directory like `data/train/cats/` and `data/train/dogs/`.
For this notebook, we will simulate loading those folders using `ImageFolder`, but we will create dummy datasets just to make the code runnable without downloading massive files.

In [ ]:
from torch.utils.data import TensorDataset
# --- SIMULATED DATA --- 
# (Replace this block with ImageFolder in the real world)
# train_dataset = datasets.ImageFolder(root='data/train', transform=train_transforms)
# val_dataset = datasets.ImageFolder(root='data/val', transform=val_transforms)

print("Simulating data (since we don't have actual images on disk)...")
dummy_train_x = torch.randn(100, 3, 224, 224)
dummy_train_y = torch.randint(0, 2, (100,))
train_dataset = TensorDataset(dummy_train_x, dummy_train_y)

dummy_val_x = torch.randn(20, 3, 224, 224)
dummy_val_y = torch.randint(0, 2, (20,))
val_dataset = TensorDataset(dummy_val_x, dummy_val_y)
# ------------------------

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## Step 4: Model Setup (Transfer Learning)
We load ResNet18, freeze it, and replace the head for binary classification (Cats vs Dogs).

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze features
for param in model.parameters():
    param.requires_grad = False
    
# Replace head
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

# Send to GPU
model = model.to(device)

# Setup Optimizer (ONLY for the new layer)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

## Step 5: Training & Validation Loop

In [ ]:
epochs = 3

for epoch in range(epochs):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {100 * correct/total:.2f}%\n")

## Phase 4 Conclusion
You have successfully mastered Computer Vision in PyTorch!
You know how to read images from disk, augment them, extract features using state-of-the-art architectures, and fine-tune them on the GPU.

This brings us to the final phase of the course. **Phase 5: Modern Deep Learning & Production**.
In the final 3 days, we will cover saving models for production (TorchScript/ONNX), writing custom PyTorch extensions, and building a mini-RNN (Recurrent Neural Network).